# MLOps end-to-end training — notebook tạm

Notebook này viết lại luồng `ingestion → validation → transformation → training → evaluation → lưu model` theo dạng tuyến tính để dễ chạy và đọc.

- Mặc định đọc file cache mới nhất trong `artifact/*/data_ingestion/feature_store/data.csv`.
- Đặt `MLOPS_DATA_SOURCE=mongo` nếu muốn đọc trực tiếp từ MongoDB.
- Không ghi connection string vào notebook; MongoDB URL phải nằm trong biến môi trường `MONGODB_URL`.
- Model được lưu dưới dạng một `sklearn Pipeline`, nên preprocessing lúc dự đoán luôn giống lúc train.
- Đây là baseline để học và kiểm chứng. Sau khi chạy ổn, từng cell có thể được ánh xạ lại vào các component production.

## 1. Import và cấu hình run

In [ ]:
from __future__ import annotations

import json
import os
import platform
import time
from datetime import datetime, timezone
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, StandardScaler

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebook":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_SOURCE = os.getenv("MLOPS_DATA_SOURCE", "cache").strip().lower()
DATABASE_NAME = os.getenv("MONGODB_DATABASE", "Prod-mlops")
COLLECTION_NAME = os.getenv("MONGODB_COLLECTION", "Proj1-Data")
TARGET_COLUMN = "Response"
TEST_SIZE = 0.25
RANDOM_STATE = 101

# Đặt MLOPS_NOTEBOOK_FAST_MODE=1 trước khi mở Jupyter để smoke test nhanh.
FAST_MODE = os.getenv("MLOPS_NOTEBOOK_FAST_MODE", "0") == "1"
MAX_ROWS = 50_000 if FAST_MODE else None
N_ESTIMATORS = 50 if FAST_MODE else 200

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
OUTPUT_DIR = PROJECT_ROOT / "artifact" / "notebook_runs" / RUN_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root : {PROJECT_ROOT}")
print(f"Data source  : {DATA_SOURCE}")
print(f"Fast mode    : {FAST_MODE}")
print(f"Output dir   : {OUTPUT_DIR}")

## 2. Data ingestion

Cell này tương ứng với `DataIngestion` và `Proj1Data`. Khi dùng cache, notebook tự chọn feature-store CSV mới nhất. Khi dùng MongoDB, nó kiểm tra kết nối bằng `ping` trước khi đọc collection.

In [ ]:
def find_latest_cached_csv(project_root: Path) -> Path:
    explicit_path = os.getenv("MLOPS_DATA_PATH")
    if explicit_path:
        path = Path(explicit_path).expanduser().resolve()
        if not path.is_file():
            raise FileNotFoundError(f"MLOPS_DATA_PATH không tồn tại: {path}")
        return path

    candidates = list(
        project_root.glob("artifact/*/data_ingestion/feature_store/data.csv")
    )
    fallback = project_root / "notebook" / "data.csv"
    if fallback.is_file():
        candidates.append(fallback)
    if not candidates:
        raise FileNotFoundError(
            "Không tìm thấy CSV cache. Hãy chạy ingestion, đặt MLOPS_DATA_PATH, "
            "hoặc đặt MLOPS_DATA_SOURCE=mongo."
        )
    return max(candidates, key=lambda path: path.stat().st_mtime)


def load_from_mongodb(database_name: str, collection_name: str) -> pd.DataFrame:
    from pymongo import MongoClient

    mongo_url = os.getenv("MONGODB_URL")
    if not mongo_url:
        raise RuntimeError("Thiếu biến môi trường MONGODB_URL.")

    client = MongoClient(mongo_url, serverSelectionTimeoutMS=10_000)
    try:
        client.admin.command("ping")
        collection = client[database_name][collection_name]
        document_count = collection.count_documents({})
        if document_count == 0:
            raise ValueError(
                f"Collection {database_name}.{collection_name} không có dữ liệu."
            )
        print(f"MongoDB documents: {document_count:,}")
        return pd.DataFrame(list(collection.find({}, {"_id": 0})))
    finally:
        client.close()


load_started = time.perf_counter()
if DATA_SOURCE == "mongo":
    raw_df = load_from_mongodb(DATABASE_NAME, COLLECTION_NAME)
    source_description = f"mongodb://{DATABASE_NAME}/{COLLECTION_NAME}"
elif DATA_SOURCE == "cache":
    data_path = find_latest_cached_csv(PROJECT_ROOT)
    raw_df = pd.read_csv(data_path)
    source_description = str(data_path)
else:
    raise ValueError("MLOPS_DATA_SOURCE chỉ nhận 'cache' hoặc 'mongo'.")

print(f"Loaded shape  : {raw_df.shape}")
print(f"Load seconds  : {time.perf_counter() - load_started:.2f}")
print(f"Source        : {source_description}")
raw_df.head()

## 3. Data validation

Validation ở đây kiểm tra tên cột, dữ liệu rỗng, target, null và duplicate. `_id` của MongoDB và `id` của dataset là định danh, được loại khỏi feature trước khi train.

In [ ]:
FEATURE_COLUMNS = [
    "Gender",
    "Age",
    "Driving_License",
    "Region_Code",
    "Previously_Insured",
    "Vehicle_Age",
    "Vehicle_Damage",
    "Annual_Premium",
    "Policy_Sales_Channel",
    "Vintage",
]
REQUIRED_COLUMNS = FEATURE_COLUMNS + [TARGET_COLUMN]

dataframe = raw_df.drop(columns=["_id", "id"], errors="ignore").copy()
missing_columns = sorted(set(REQUIRED_COLUMNS) - set(dataframe.columns))
extra_columns = sorted(set(dataframe.columns) - set(REQUIRED_COLUMNS))
target_values = sorted(dataframe[TARGET_COLUMN].dropna().unique().tolist()) if TARGET_COLUMN in dataframe else []

validation_report = {
    "status": not missing_columns and not dataframe.empty and set(target_values) == {0, 1},
    "row_count": int(len(dataframe)),
    "column_count": int(len(dataframe.columns)),
    "missing_columns": missing_columns,
    "extra_columns": extra_columns,
    "null_counts": {key: int(value) for key, value in dataframe.isna().sum().items() if value},
    "duplicate_rows": int(dataframe.duplicated().sum()),
    "target_values": target_values,
}

(OUTPUT_DIR / "validation_report.json").write_text(
    json.dumps(validation_report, indent=2), encoding="utf-8"
)
print(json.dumps(validation_report, indent=2))

if not validation_report["status"]:
    raise ValueError("Data validation thất bại; xem validation_report ở trên.")

if extra_columns:
    print(f"Cảnh báo: bỏ qua extra columns: {extra_columns}")
dataframe = dataframe[REQUIRED_COLUMNS].dropna().drop_duplicates().reset_index(drop=True)

if MAX_ROWS is not None and len(dataframe) > MAX_ROWS:
    dataframe, _ = train_test_split(
        dataframe,
        train_size=MAX_ROWS,
        stratify=dataframe[TARGET_COLUMN],
        random_state=RANDOM_STATE,
    )
    dataframe = dataframe.reset_index(drop=True)
    print(f"Fast mode sample: {len(dataframe):,} rows")

dataframe[TARGET_COLUMN].value_counts(normalize=False).sort_index()

## 4. Train/test split

Split dùng `stratify` để tỷ lệ `Response=0/1` ổn định giữa train và test. Mọi transformer chỉ được `fit` trên train để tránh data leakage.

In [ ]:
X = dataframe[FEATURE_COLUMNS]
y = dataframe[TARGET_COLUMN].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Train: {X_train.shape}, positive rate={y_train.mean():.4f}")
print(f"Test : {X_test.shape}, positive rate={y_test.mean():.4f}")

## 5. Data transformation và model

`ColumnTransformer` xử lý numeric/categorical theo tên cột. `OneHotEncoder(handle_unknown='ignore')` giúp inference không vỡ khi gặp category chưa thấy. Baseline dùng `class_weight='balanced_subsample'` thay vì SMOTEENN để chạy nhanh trên 381k dòng và tuyệt đối không resample test set.

In [ ]:
standard_scaled_columns = ["Age", "Vintage"]
minmax_scaled_columns = ["Annual_Premium"]
other_numeric_columns = [
    "Driving_License",
    "Region_Code",
    "Previously_Insured",
    "Policy_Sales_Channel",
]
categorical_columns = ["Gender", "Vehicle_Age", "Vehicle_Damage"]

standard_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
minmax_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", MinMaxScaler()),
])
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("one_hot", OneHotEncoder(handle_unknown="ignore", drop="if_binary")),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("standard", standard_pipeline, standard_scaled_columns),
        ("minmax", minmax_pipeline, minmax_scaled_columns),
        ("numeric", numeric_pipeline, other_numeric_columns),
        ("categorical", categorical_pipeline, categorical_columns),
    ],
    remainder="drop",
)

classifier = RandomForestClassifier(
    n_estimators=N_ESTIMATORS,
    min_samples_split=7,
    min_samples_leaf=6,
    max_depth=10,
    criterion="entropy",
    class_weight="balanced_subsample",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

model_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", classifier),
])
model_pipeline

## 6. Model training

In [ ]:
train_started = time.perf_counter()
model_pipeline.fit(X_train, y_train)
training_seconds = time.perf_counter() - train_started
print(f"Training completed in {training_seconds:.2f} seconds")

## 7. Model evaluation

Với bài toán mất cân bằng, không nên chỉ nhìn accuracy. F1, precision, recall, ROC-AUC và confusion matrix cho biết rõ hơn khả năng nhận diện lớp `Response=1`.

In [ ]:
y_pred = model_pipeline.predict(X_test)
y_probability = model_pipeline.predict_proba(X_test)[:, 1]

metrics = {
    "accuracy": float(accuracy_score(y_test, y_pred)),
    "precision": float(precision_score(y_test, y_pred, zero_division=0)),
    "recall": float(recall_score(y_test, y_pred, zero_division=0)),
    "f1": float(f1_score(y_test, y_pred, zero_division=0)),
    "roc_auc": float(roc_auc_score(y_test, y_probability)),
    "training_seconds": float(training_seconds),
    "train_rows": int(len(X_train)),
    "test_rows": int(len(X_test)),
}

print(json.dumps(metrics, indent=2))
print("\nClassification report:\n")
print(classification_report(y_test, y_pred, digits=4, zero_division=0))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred, cmap="Blues")
plt.title("Random Forest — confusion matrix")
plt.show()

## 8. Lưu và reload artifact

Lưu toàn bộ pipeline thay vì lưu preprocessor/model rời nhau. Cell cuối reload model và chạy thử vài dòng để bảo đảm artifact dùng được.

In [ ]:
model_path = OUTPUT_DIR / "model.joblib"
metrics_path = OUTPUT_DIR / "metrics.json"
run_config_path = OUTPUT_DIR / "run_config.json"

run_config = {
    "run_id": RUN_ID,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "source": source_description,
    "target_column": TARGET_COLUMN,
    "test_size": TEST_SIZE,
    "random_state": RANDOM_STATE,
    "n_estimators": N_ESTIMATORS,
    "fast_mode": FAST_MODE,
    "python_version": platform.python_version(),
    "pandas_version": pd.__version__,
    "sklearn_version": sklearn.__version__,
}

joblib.dump(model_pipeline, model_path)
metrics_path.write_text(json.dumps(metrics, indent=2), encoding="utf-8")
run_config_path.write_text(json.dumps(run_config, indent=2), encoding="utf-8")

reloaded_model = joblib.load(model_path)
sample_predictions = reloaded_model.predict(X_test.head(5))

print(f"Model  : {model_path}")
print(f"Metrics: {metrics_path}")
print(f"Config : {run_config_path}")
print(f"Reload smoke-test predictions: {sample_predictions.tolist()}")

## Mapping notebook sang hệ thống hiện tại

| Notebook | Component trong `src` | Artifact chính |
|---|---|---|
| Data ingestion | `DataIngestion`, `Proj1Data` | DataFrame / CSV cache |
| Data validation | `DataValidation` | `validation_report.json` |
| Split + preprocessing | `DataTransformation` | fitted preprocessor |
| Model training | `ModelTrainer` | fitted Random Forest |
| Model evaluation | `ModelEvaluation` | `metrics.json` |
| Save/reload | model registry/pusher | `model.joblib` |

Khác biệt có chủ đích so với code hiện tại:

1. Preprocessing và model nằm trong cùng một pipeline để inference không phải lặp lại mapping thủ công.
2. `OneHotEncoder` chỉ fit trên train và xử lý category mới an toàn.
3. Không resample test set. Test set phải đại diện cho phân phối dữ liệu thật.
4. Baseline dùng class weight; SMOTE/SMOTEENN chỉ nên thử bên trong train/cross-validation và đo lại bằng cùng test set nguyên bản.
5. Mỗi run lưu model, metrics, config và validation report trong cùng một thư mục để truy vết.